# The Cell State Is a Gradient Highway

**Topic:** LSTM gating &nbsp;·&nbsp; **Difficulty:** Hard

An LSTM's cell state update is `c_t = f_t * c_{t-1} + i_t * g_t`. The path from
`c_{t-1}` to `c_t` is multiplication by the forget gate (no weight matrix, no
`tanh` derivative), so the gradient along that path is just a product of forget
gates. Keep them near 1 and the gradient walks 100 steps unharmed.

Use the setup cell's weights: `f_t = sigmoid(Wf @ [h_{t-1}; x_t] + bf)`,
`i_t = sigmoid(Wi @ [h_{t-1}; x_t])`, `g_t = tanh(Wg @ [h_{t-1}; x_t])` and
`h_t = tanh(c_t)`, where `[h_{t-1}; x_t]` stacks the 24 hidden entries on the
6 input entries (30 total) and only the forget gate gets a bias.

Implement both recurrences and compare the gradient reaching step 0.

## The data (created in the next cell)

- Hidden size 24, sequence length 100.
- Vanilla RNN: `h_t = tanh(W h_{t-1} + U x_t)` with spectral radius `0.9`.
- LSTM cell with the forget-gate bias initialized to `+2.0` (so gates start near
  `sigmoid(2) = 0.88`), the standard trick.

## Your task

With **NumPy only**:

1. Run the vanilla RNN for 100 steps and backpropagate `dL/dh_100 = ones`,
   recording the norm at each backprop step, so `rnn_norms[t]` holds
   `||dL/dh_{t+1}||` (the entry at `t = 99` is `||dL/dh_100||`).
2. Run an LSTM for 100 steps. Backpropagate along the **cell path only**:
   `dL/dc_{t-1} = dL/dc_t * f_t`, starting from `dL/dc_100 = ones`. Record the
   norms with the same convention, so `lstm_norms[t]` holds `||dL/dc_{t+1}||`.
3. Set `answers` to a dict with `"rnn_norms"` `(100,)`, `"lstm_norms"` `(100,)`,
   `"mean_forget_gate"`, `"rnn_ratio"` and `"lstm_ratio"`: the norm at `t = 0`
   divided by the norm at `t = 99` for each.

## Evaluation

Compared to the reference to a tolerance of 1e-6 times (1 plus the reference
value). The evaluation cell checks the mean forget gate is above 0.75 and that
the LSTM retains at least a million times more gradient at step 0.

In [ ]:
import numpy as np

rng = np.random.default_rng(4003)
H, T, D = 24, 100, 6
W = rng.normal(0, 1, (H, H))
W = 0.9 * W / np.max(np.abs(np.linalg.eigvals(W)))
U = rng.normal(0, 0.4, (H, D))

Wf = rng.normal(0, 0.3, (H, H + D))
bf = np.full(H, 2.0)                 # the standard positive forget bias
Wi = rng.normal(0, 0.3, (H, H + D))
Wg = rng.normal(0, 0.3, (H, H + D))
X = rng.normal(0, 1, (T, D))
print(f"hidden {H}, {T} steps, forget-gate bias {bf[0]:+.1f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("rnn_norms", [100], 11.21022753619897, -4.930796084696997, 1e-06),
    ("lstm_norms", [100], 55.76385975771934, -1.8976778585671719, 1e-06),
    ("mean_forget_gate", [], 0.7991510704351739, -0.1338082687705475, 1e-06),
    ("rnn_ratio", [], 5.096319087955352e-28, -8.533175509484831e-29, 1e-06),
    ("lstm_ratio", [], 0.002403284633320505, -0.00040240120803737686, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "lstm_ratio" in answers:
    _win = answers["lstm_ratio"] > 1e6 * answers["rnn_ratio"]
    _gate = answers["mean_forget_gate"] > 0.75
    print(f"  {'cell path keeps gradient':<24} {'ok' if _win else 'wrong'}  "
          f"(lstm {answers['lstm_ratio']:.3e} vs rnn {answers['rnn_ratio']:.3e})")
    print(f"  {'forget gates near 1':<24} {'ok' if _gate else 'wrong'}  "
          f"(mean {answers['mean_forget_gate']:.3f})")
    _ok = _ok and _win and _gate

print("=" * 56)
if _ok:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

The vanilla path multiplies by `W^T diag(1 - h^2)` every step, and both factors
shrink it. The LSTM's cell path multiplies by the forget gate alone, and with a
`+2.0` bias those gates average about 0.8, and the units whose gates stay
closest to 1 dominate the norm, so 100 steps still costs a factor of about
`2 x 10^-3`, which is small but astronomically larger than the vanilla
product. That is
also why the positive forget bias is a standard initialization: at bias 0 the
gates start near 0.5 and 100 steps costs `2^-100`.

```python
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

h = np.zeros(H)
hs = []
for t in range(T):
    h = np.tanh(W @ h + U @ X[t])
    hs.append(h.copy())
g = np.ones(H)
rnn_norms = np.zeros(T)
for t in reversed(range(T)):
    rnn_norms[t] = float(np.linalg.norm(g))
    g = W.T @ (g * (1 - hs[t] ** 2))

hL = np.zeros(H)
c = np.zeros(H)
fgates = []
for t in range(T):
    z = np.concatenate([hL, X[t]])
    f = sigmoid(Wf @ z + bf)
    i = sigmoid(Wi @ z)
    gg = np.tanh(Wg @ z)
    c = f * c + i * gg
    hL = np.tanh(c)
    fgates.append(f.copy())

gc = np.ones(H)
lstm_norms = np.zeros(T)
for t in reversed(range(T)):
    lstm_norms[t] = float(np.linalg.norm(gc))
    gc = gc * fgates[t]

answers = {
    "rnn_norms": rnn_norms,
    "lstm_norms": lstm_norms,
    "mean_forget_gate": float(np.mean(fgates)),
    "rnn_ratio": float(rnn_norms[0] / rnn_norms[-1]),
    "lstm_ratio": float(lstm_norms[0] / lstm_norms[-1]),
}
```